# Chapter 3 · Lab 2 — Implement and integrate optimizations

Prerequisites: Lab 1's ranked evidence table, Chapter 1's model/cache/checkpoint
contract, and Chapter 2's timing helpers. Read the [fusion and CuTe theory](../background.md#fusion-from-tensor-shapes-and-rounding-boundaries).
Run top to bottom from a fresh kernel. Reusable outputs are
[cute_kernels.py](cute_kernels.py) and [model_opt.py](model_opt.py), imported by Chapter 4.
The supplied mechanisms are complete runnable references; your exercises modify
tile size, work partition or fusion boundary and test the predicted effect.

In [ ]:
from pathlib import Path
import importlib, json, os, sys, time
from uuid import uuid4
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'pyproject.toml').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
# Keep compiler disk caches from different DSL versions separate. In-memory JIT reuse remains enabled.
os.environ.setdefault('CUTE_DSL_DISABLE_FILE_CACHING', '1')
from IPython.display import display, Markdown, Image
from shared import performance as perf
study = importlib.import_module('chapters.03_kernels.code.study')
plots = importlib.import_module('chapters.03_kernels.code.plots')
RUN_GPU = True
MODEL_KEYS = ['8b', '32b']
EXTERNAL_PROFILERS = True
# Set P03_REPLAY_DIR only to inspect an existing run. Replay never creates measurements.
REPLAY_DIR = os.environ.get('P03_REPLAY_DIR')

## 1. Establish toolchain and thread/value ownership

Use DSL **4.2.1**, the exact official example commit
`f3fde58372d33e9a5650ba7b80fc48b3b49d40c8`,
`examples/python/CuTeDSL/ampere/elementwise_add.py --M 3 --N 12`.
The [setup guide](../../../shared/SETUP.md#chapter-3-verified-cute-environment)
records verified GB10/SM121 versions. Run the example before course kernels when
changing the toolchain. Do not port Hopper-only instructions to GB10 by assumption.

A layout is shape + element strides. In a four-warp block, warp `w` owns row
`4*block+w`; lane `l` owns columns `l+32*i`. Read `warp_sum` and draw its five
butterfly exchanges. Predication must be uniform within a warp at those exchanges.
Registers hold partial sums; no inter-warp shared data requires a block barrier.
Explain where two barriers become necessary if a key tile is staged in shared memory.
Dynamic shapes reuse compiled functions; widths and tile/work partition are JIT
specializations. The host passes PyTorch's current stream explicitly.

In [ ]:
import inspect
model_opt = importlib.import_module('chapters.03_kernels.code.model_opt')
kernels = model_opt.kernels()
print(study.manifest('toolchain')['versions'])
print(inspect.getsource(kernels.warp_sum))
print(inspect.getsource(kernels._launch))

## 2. Predict → modify → validate each fusion

| Mechanism | Input → output | Guided implementation experiment |
|---|---|---|
| RMSNorm | BF16 `[...,D]`, weights `[D]` → same shape | Change rows per block; predict register pressure versus parallelism at D=4096/5120. Keep FP32 reduction and intermediate BF16 rounding. |
| SwiGLU | BF16 gate/up `[B,T,I]` → same shape | Compare 128 and 256 threads; explain why one fewer launch and `2*B*T*I*2` bytes need not dominate small decode. |
| Q/K norm + RoPE | `[B,T,Hq/Hkv,128]`, absolute `[B,T]` positions → `[B,Hq/Hkv,T,128]` | Separate Q and K launches, then restore fusion. Preserve rotate-half and BF16 product rounding; measure both launch and traffic changes. |
| Causal GQA prefill | Q `[B,Hq,T,128]`, KV `[B,Hkv,P+T,128]` → Q shape | Change query warps/block 1→4 or key tile 17→32. Predict useful parallelism, tail work and KV rereads. |
| GQA decode | Q `[B,Hq,1,128]`, appended KV → Q shape | Compare one versus four warps/block. For one query three warps are idle. Explain why context splitting may help and what merge traffic it adds. |

All kernels store BF16 and reduce/accumulate in FP32. Attention selects the KV
head directly, never expands heads or writes scores. This first SIMT implementation
streams key tiles and retains four Q/output coordinates per lane. It recomputes
QK in three tile passes (maximum, probability mass, weighted V) to avoid score
storage; normalized online output is rescaled once per tile. Account for these
extra FLOPs and K reads when explaining performance. It does not use
tensor-core MMA or shared-memory reuse; an explained slowdown is valid.

**Online-softmax exercise:** inspect [lab.py](lab.py)'s FP32 oracle, then implement
a two-group `(m,ell,u)` merge. Show a counterexample to averaging normalized group
outputs. The oracle repeats KV for clarity and is explicitly not a fused kernel.

In [ ]:
# Read complete implementations, then edit the module and restart the kernel for each variant.
for function in [kernels.rms_kernel, kernels.swiglu_kernel, kernels.qk_rope_kernel, kernels.attention_kernel]:
    print(inspect.getsource(function))

## 3. Freeze tolerances, then check mechanisms before timing

Operator acceptance: `atol=0.02, rtol=0.02` versus the BF16 baseline or FP32
attention oracle. Small fixture logits: `atol=0.03, rtol=0.02`; real checkpoint
logits: `atol=0.25, rtol=0.02`, inherited from Chapter 2. Tolerances are declared
before performance comparisons; investigate failures instead of relaxing them.
Tests cover both widths/head counts, partial key tiles, extreme scores, transposed
projection layouts, contiguous cache, nonzero prefixes and cache ownership.
An operator test is not evidence that a real model passed: the next stage also
compares all logits on a fixed real-weight fixture for both full checkpoints.

Integration regressions are especially instructive: compile broadcast and
non-broadcast position layouts in both orders, and require bitwise rotary agreement
when RMS sums are exact. Inspect `round_bf16`: replacing it with casts permits the
pinned toolchain to contract a product and sum, changing the intended rounding.
Predict the effect, reproduce the failing local check, then restore the boundary.

Compare Q/K’s four-consecutive-value reduction with the strided RMS kernel.
Explain why changing FP32 addition order can change a BF16 rounding decision.
Inspect `scaled_score`: preserving FP32 score scaling before subtraction is
another rounding boundary. Retest full-model logits after either change.

In [ ]:
import subprocess
if RUN_GPU and not REPLAY_DIR:
    completed = subprocess.run([sys.executable,'-m','unittest','discover','-s','tests',
                                '-p','test_optimized_kernels.py','-v'],cwd=ROOT,check=True)
else:
    print('Operator suite not rerun in replay/reading mode; inspect saved validation evidence.')

### Measure one local work-partition change

This focused experiment compares SwiGLU blocks 128/256 at both MLP widths.
Correctness runs first. The **kernel** boundary prebuilds DLPack descriptors and
output storage; the **adapter** boundary includes allocation and dispatch; the
**reference** boundary is PyTorch SiLU plus multiplication. CUDA event intervals
may include launch latency. These synthetic component measurements supplement,
and do not replace, the real-model integrated timings below.

In [ ]:
if RUN_GPU and not REPLAY_DIR:
    component_dir = ROOT/'results'/('p03-component-' + uuid4().hex[:8])
    components = importlib.import_module('chapters.03_kernels.code.component_bench')
    component_rows = components.run(component_dir)
    display(component_rows)
    print('Component artifacts:', component_dir)

## 4. Integrate through inherited operator methods

`OptimizedQwen3(config, optimizations=Optimizations(...), strict=True)` inherits
Chapter 1's forward path. Parameter names, checkpoint mapping, GEMMs, residuals,
logits selection and cache concatenation are unchanged. Cache is caller-owned;
append allocates new tensors. `decode=True` selects only final logits even for
prefill. Input length chooses prefill versus decode attention.

Switches independently control RMSNorm, SwiGLU, Q/K+RoPE, prefill attention and
decode attention. `strict=False` records unsupported reference fallback for
exploration; the driver uses `strict=True` and rejects any unexpected fallback.
Compilation failures are errors, not fallback. Disabled switches are intentional
ablations. Ragged/paged inputs need Chapter 4's adapter, not silent padding.

In [ ]:
print(inspect.signature(model_opt.OptimizedQwen3))
display({name:vars(flags) for name,flags in model_opt.PRESETS.items()})
variant = model_opt.Optimizations(key_tile=17, attention_warps=1, activation_block=128)
print('Exercise variant, not selected by the default sweep:', variant)
# Add a named preset in model_opt.py, restart, and save it to a fresh run directory.

## 5. Repeat the matched profiling workflow

Run all optimized cases from Lab 1; fusion-only and attention-only ablations use
batch 1 / prompt 2048 for both models and phases. Each case first takes unprofiled
timings, then separate PyTorch/Nsight captures. Confirm custom kernel names in
`kernel_summary.json` and zero unexpected fallbacks in strict status records.
Measure kernel counts, intermediate allocations, launch gaps and integrated latency.
Counter evidence remains incomplete when the saved Nsight Compute log reports
`ERR_NVGPUCTRPERM`. No missing counters are inferred from modeled bytes.

In [ ]:
if RUN_GPU:
    if REPLAY_DIR:
        RUN_DIR = Path(REPLAY_DIR).resolve()
        print('Replaying saved evidence; no new measurements:', RUN_DIR)
        assert (RUN_DIR/'completion.json').is_file()
    else:
        RUN_DIR = ROOT/'results'/('p03-optimized-' + time.strftime('%Y%m%d-%H%M%S') + '-' + uuid4().hex[:6])
        study.execute(RUN_DIR, 'optimized', MODEL_KEYS, external=EXTERNAL_PROFILERS)
    raw = plots.read_rows(RUN_DIR/'results.csv')
    summary = perf.summarize_model_rows(raw)
    display(Markdown(f'Artifacts: `{RUN_DIR}` · {len(raw)} unprofiled observations'))
    display(json.loads((RUN_DIR/'status.json').read_text()))
else:
    print('GPU/checkpoint work unmeasured. Enable RUN_GPU after setup.')

## Completion and explanation

Save your changed kernel, frozen prediction and correctness results before the
variant timings. Explain one observed tile/work-partition result and a failed
hypothesis. Compare fusion-only, attention-only and all-enabled measurements with
Lab 1 using the **same model, batch, context and boundary**. Keep compilation
metadata and verify every specialization warmed before measured repeats.

Report kernel-only, adapter-inclusive and full-model costs for your chosen local
change; DLPack dispatch and allocation belong to the integrated path. Apply Amdahl
with the baseline fraction from Lab 1, then explain disagreement using traces.
A smaller intermediate or fewer launches alone is not a performance conclusion.
Continue to [Lab 3](lab3.ipynb) for the complete matched sweep.